In [1]:
import pandas as pd
import sys
sys.path.append('..')
from src.data_loader import load_orders

orders = load_orders()

#Work only for prior orders for features
prior_orders = orders[orders['eval_set'] == 'prior'].copy()

#User order summary
user_summary = (
    prior_orders.groupby('user_id')
    .agg(
        total_orders=('order_id', 'count'),
        max_order_number=('order_number', 'max'),
        avg_days_between=('days_since_prior_order', 'mean'),
        std_days_between=('days_since_prior_order', 'std'),
        favourite_dow=('order_dow', lambda x: x.mode()[0]),
        favourite_hour=('order_hour_of_day', lambda x: x.mode()[0]),
    )
   .reset_index()
)

#Fill Std nulls (user with only 1 order has no std)
user_summary['std_days_between'] = user_summary['std_days_between'].fillna(0)

print(f"User summary shape: {user_summary.shape}")
print(user_summary.head(5).to_string())
print()
print(user_summary.describe())

#Save as parquet for later use
user_summary.to_parquet('../data/processed/user_summary.parquet', index=False)
print("User summary saved to ../data/processed/user_summary.parquet")

User summary shape: (206209, 7)
   user_id  total_orders  max_order_number  avg_days_between  std_days_between  favourite_dow  favourite_hour
0        1            10                10         19.555556          9.395625              1               7
1        2            14                14         15.230769          9.867065              1              10
2        3            12                12         12.090909          5.375026              0              16
3        4             5                 5         13.750000          9.500000              4              11
4        5             4                 4         13.333333          4.932883              3              18

             user_id   total_orders  max_order_number  avg_days_between  \
count  206209.000000  206209.000000     206209.000000     206209.000000   
mean   103105.000000      15.590367         15.590367         15.209435   
std     59527.555167      16.654774         16.654774          7.105277   
min    